In [1]:
# 批量处理长聊天记录并生成记忆摘要

import os
import sys
import json
import time
import shutil
import importlib

# --- 环境变量设置 ---
print(f"DEEPSEEK_API_KEY: {'Yes' if os.environ.get('DEEPSEEK_API_KEY') else 'No'}")

# 添加项目根目录到路径
root_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

import utils.llm.llm_utils
importlib.reload(utils.llm.llm_utils)

from utils.llm.llm_utils import (
    summarize_memory, 
    save_memory, 
    get_memory_file_path, 
    load_memory_config,
    load_memory,
    MEMORY_DIR
)
from utils.llm.llm_send import load_history, save_history
from utils.llm.text_utils import resolve_path, load_text, load_json_file


# 加载聊天列表
chat_list_path = os.path.join(MEMORY_DIR, "..", "chat_list.json")
if not os.path.exists(chat_list_path):
    # Fallback if MEMORY_DIR is not set correctly relative to chat_list
    chat_list_path = os.path.join(root_dir, "utils", "llm", "dialogue_histories", "chat_list.json")

print(f"Loading chat list from: {chat_list_path}")
chat_list = load_json_file(chat_list_path)
chats_to_process = list(chat_list.keys())
chats_to_process = ["default"]
# chats_to_process = ["guitar"] # Uncomment to test specific chat

print(f"Found {len(chats_to_process)} chats: {chats_to_process}")

# 加载 System Prompt
try:
    sys_prompt_path = os.path.join(root_dir, "config_files", "sys_prompt.txt")
    system_prompt = load_text(sys_prompt_path)
    print(f"✅ SysPrompt加载成功")
except Exception as e:
    print(f"⚠️ SysPrompt加载失败: {e}")
    system_prompt = "你是一个专业的助手。"

config = load_memory_config()
max_history = config.get("max_history_length", 20)
retain_history = config.get("retain_history_length", 5)

for chatname in chats_to_process:
    print(f"\n{'='*40}")
    print(f"🚀 开始处理: {chatname}")
    print(f"{'='*40}")

    mem_path = get_memory_file_path(chatname)
    
    # --- 修复：检查并移动错误位置的记忆文件 ---
    possible_wrong_paths = [
        os.path.join(root_dir, f"{chatname}_memory.json"),
        os.path.join(os.path.dirname(MEMORY_DIR), f"{chatname}_memory.json"),
    ]

    for wrong_path in possible_wrong_paths:
        if os.path.exists(wrong_path) and os.path.abspath(wrong_path) != os.path.abspath(mem_path):
            try:
                shutil.move(wrong_path, mem_path)
                print(f"✅ 已移动错误文件: {wrong_path}")
            except: pass

    try:
        path, full_history = load_history(chatname)
        print(f"历史总条数: {len(full_history)}")

        if len(full_history) <= max_history:
            print("✅ 历史记录较短，无需处理。")
        else:
            processed_count = 0
            history_to_process = list(full_history)
            
            # 循环处理直到剩余历史小于最大限制
            while len(history_to_process) > max_history:
                # chunk_to_process = history_to_process[:max_history] # 这一行是多余的，下面已经定义了 chunk
                
                print(f"处理第 {processed_count + 1} 组 (剩余 {len(history_to_process)} 条)...")
                
                # current_mem = load_memory(chatname) # 这一行也是多余的，summarize_memory 内部会处理
                
                # 这里的逻辑是：每次取 max_history 长度的数据去总结
                # summarize_memory 内部逻辑是：如果长度 > retain_history_length，则总结前面的，保留后面的
                # 所以我们传入 max_history 长度的数据，期望它总结掉 (max_history - retain_history_length) 条
                chunk = history_to_process[:max_history]
                
                # 调用 summarize_memory
                # 注意：summarize_memory 现在返回的是 (recent_history, success)
                # recent_history 是 chunk 中被保留下来的部分
                recent, success = summarize_memory(chatname, chunk, system_prompt)
                
                if success:
                    processed_count += 1
                    # 计算被概括掉的数量 = 原chunk长度 - 保留的长度
                    summarized_count = len(chunk) - len(recent)
                    
                    # 从待处理列表中移除已经被概括掉的部分
                    # 注意：history_to_process 是完整的列表，我们需要移除头部已经被概括的部分
                    # 这里的逻辑稍微有点绕：
                    # chunk 是 history_to_process 的前 max_history 条
                    # recent 是 chunk 的后 retain_history_length 条
                    # 所以被移除的是 chunk 的前 (len(chunk) - len(recent)) 条
                    # 这些被移除的条目对应 history_to_process 的头部
                    history_to_process = history_to_process[summarized_count:]
                    
                    # 【关键修复】保存更新后的历史记录回文件！
                    # 如果不保存，下次循环或者下次运行脚本时，文件里的内容还是旧的
                    save_history(chatname, history_to_process)
                    print(f"  ✅ 已更新历史记录文件 (剩余 {len(history_to_process)} 条)")
                    
                    time.sleep(1)
                else:
                    print("  - 概括未执行（可能失败或无需概括）")
                    break
                    
            print(f"✅ {chatname} 处理完成！共生成 {processed_count} 个新片段。")

        final_memory = load_memory(chatname)
        print(f"最终记忆片段数: {len(final_memory)}")
        if final_memory:
            print(f"最新记忆: {final_memory[-1].get('summary', '')[:50]}...")

    except Exception as e:
        print(f"❌ 处理 {chatname} 时发生错误: {e}")
        import traceback
        traceback.print_exc()


DEEPSEEK_API_KEY: Yes
Loading chat list from: d:\服务器项目测试\interact_metahuman_with_neurosync\utils\llm\dialogue_histories\memories\..\chat_list.json
Found 1 chats: ['default']
d:\服务器项目测试\interact_metahuman_with_neurosync\config_files\sys_prompt.txt
✅ SysPrompt加载成功

🚀 开始处理: default
d:\服务器项目测试\interact_metahuman_with_neurosync\utils\llm\dialogue_histories\default.txt
d:\服务器项目测试\interact_metahuman_with_neurosync\utils\llm\dialogue_histories\default.txt
历史总条数: 60
处理第 1 组 (剩余 60 条)...
🧹 开始概括对话历史 (当前条数: 20)...
📡 调用 deepseek-chat 提取记忆片段...
Loading chat list from: d:\服务器项目测试\interact_metahuman_with_neurosync\utils\llm\dialogue_histories\memories\..\chat_list.json
Found 1 chats: ['default']
d:\服务器项目测试\interact_metahuman_with_neurosync\config_files\sys_prompt.txt
✅ SysPrompt加载成功

🚀 开始处理: default
d:\服务器项目测试\interact_metahuman_with_neurosync\utils\llm\dialogue_histories\default.txt
d:\服务器项目测试\interact_metahuman_with_neurosync\utils\llm\dialogue_histories\default.txt
历史总条数: 60
处理第 1 组 (剩余 60 条)...
🧹 